# 01 — Exploring the disease.sh data

**Goal:** sanity-check the data we pulled from disease.sh, look at COVID trajectories for a few large countries, and inspect the engineered features.

Prerequisites (run from project root):
```bash
python -m src.data.fetch
python -m src.features.build
```

In [ ]:
import sys
from pathlib import Path
if '..' not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
DATA = Path('../data/processed')

In [ ]:
historical = pd.read_parquet(DATA / 'historical.parquet')
historical['date'] = pd.to_datetime(historical['date'])
snapshots = pd.read_parquet(DATA / 'snapshots.parquet')
features = pd.read_parquet(DATA / 'features.parquet')
features['date'] = pd.to_datetime(features['date'])

print(f'Historical: {len(historical):,} rows × {historical["country"].nunique()} countries')
print(f'Snapshots:  {len(snapshots)} countries')
print(f'Features:   {len(features):,} rows × {features.shape[1]} cols')
print(f'Date range: {features["date"].min():%Y-%m-%d} → {features["date"].max():%Y-%m-%d}')

## Trajectory for a few large countries

How did the pandemic unfold differently across countries with different demographics and policies?

In [ ]:
focus = ['India', 'USA', 'UK', 'Brazil', 'Japan', 'Germany']
available = [c for c in focus if c in features['country'].values]
print('Plotting:', available)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
sub = features[features['country'].isin(available)]
for ax, metric, ylabel in [
    (axes[0], 'new_cases_smoothed', 'Daily new cases (7-day mean)'),
    (axes[1], 'cases_per_million', 'Daily new cases per million'),
]:
    for country, grp in sub.groupby('country'):
        ax.plot(grp['date'], grp[metric], label=country, lw=1.5, alpha=0.8)
    ax.set_ylabel(ylabel)
    ax.legend(ncol=3, loc='upper right')
axes[0].set_title('COVID-19 trajectories', fontsize=13, fontweight='bold')
axes[1].set_xlabel('')
plt.tight_layout()
plt.show()

## R-effective over time — when did each wave start to recede?

R-effective above 1 means the outbreak is growing; below 1 means receding. Sustained crossings of the 1 line mark inflection points.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for country, grp in sub.groupby('country'):
    ax.plot(grp['date'], grp['r_effective_approx'].clip(0, 3), label=country, lw=1.2, alpha=0.7)
ax.axhline(1, color='red', ls='--', alpha=0.5, label='R = 1 (epidemic threshold)')
ax.set(title='Effective reproduction number over time',
       xlabel='', ylabel='R-effective (approx, clipped to 3)')
ax.legend(ncol=3)
plt.tight_layout()
plt.show()

## Static portrait: cases per million by continent

In [ ]:
continent_totals = snapshots.dropna(subset=['continent', 'cases_per_million']).copy()
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=continent_totals,
    x='continent', y='cases_per_million',
    ax=ax, palette='Set2', showfliers=False,
)
ax.set(title='Cumulative COVID cases per million population by continent',
       xlabel='', ylabel='Cases per million')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print('Median cases per million by continent:')
print(continent_totals.groupby('continent')['cases_per_million'].median().sort_values(ascending=False).round(0))

## Next: train models and explore the dashboard

```bash
python -m src.models.train --models all
streamlit run app.py
```